# Week 2 — Task 4: Channel Effectiveness — Run the Experiment Yourself

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Datasets:** `channel_trials.csv` (30 rows, 5 channels × 6 ratios) and `iris.csv`.

**All charts built with `fig, ax = plt.subplots()`. OO API throughout.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

In [ ]:
trials = pd.read_csv('data/channel_trials.csv')
print(f'Shape: {trials.shape}')
print(f'Channels: {trials["channel"].unique()}')
print(f'True ratios: {sorted(trials["true_ratio"].unique())}')
trials.head(10)

## Step 1 — Five Encoding Functions (One Per Channel)

No axis labels, no tick values, no numbers anywhere — the reader must judge from the channel alone.

In [ ]:
def draw_position(ax, large, small):
    """Two dots on a common scale."""
    ax.scatter([0, 1], [large, small], s=80, color='#2166ac', zorder=3)
    ax.axhline(0, color='#888888', lw=0.8, linestyle='--')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(0, max(large, small) * 1.2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')

def draw_length(ax, large, small):
    """Two bars from a common zero baseline."""
    ax.bar([0, 1], [large, small], color='#2166ac', width=0.4, alpha=0.85)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(0, max(large, small) * 1.2)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

def draw_angle(ax, large, small):
    """Two wedges of a pie."""
    total = large + small
    ax.pie([large, small], colors=['#2166ac', '#d73027'], startangle=90,
           wedgeprops=dict(edgecolor='white', lw=1.5))
    ax.set_aspect('equal')

def draw_area(ax, large, small):
    """Two circles — area proportional to value (s=area, not radius)."""
    # s is area in points^2
    scale = 800 / large  # normalise so large circle has area 800
    ax.scatter([0.3, 0.7], [0.5, 0.5],
               s=[large * scale, small * scale],
               color='#2166ac', alpha=0.75, edgecolors='none')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    # Note: s is proportional to value (area = value * scale) — this is correct.
    # If s were proportional to value^2 that would be an area lie.

def draw_colour(ax, large, small):
    """Two squares — lightness proportional to value (sequential ramp)."""
    max_v = max(large, small)
    l_light = 1 - large / max_v * 0.8   # more value = darker
    s_light = 1 - small / max_v * 0.8
    color_large = mcolors.hsv_to_rgb([0.6, 0.6, l_light])
    color_small = mcolors.hsv_to_rgb([0.6, 0.6, s_light])
    ax.add_patch(mpatches.Rectangle((0.05, 0.1), 0.35, 0.8, color=color_large))
    ax.add_patch(mpatches.Rectangle((0.6,  0.1), 0.35, 0.8, color=color_small))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

In [ ]:
# Verify: show trial #1 encoded five ways
trial = trials[trials['trial'] == 1].iloc[0]
large_val, small_val = trial['value_large'], trial['value_small']

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
draw_position(axes[0], large_val, small_val)
draw_length(  axes[1], large_val, small_val)
draw_angle(   axes[2], large_val, small_val)
draw_area(    axes[3], large_val, small_val)
draw_colour(  axes[4], large_val, small_val)

for ax, title in zip(axes, ['Position', 'Length', 'Angle', 'Area', 'Colour (lightness)']):
    ax.set_title(title, fontsize=9)

fig.suptitle(f'Same pair ({large_val}, {small_val}) encoded five ways — no numbers shown to reader',
             fontsize=10)
plt.tight_layout()
plt.savefig('task4_encoding_verification.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Run All 30 Trials on a Real Person

**Subject:** Bilal (classmate)  
**Protocol:** Trials shuffled. Each chart shown without numbers. Question: 'The smaller is what percentage of the larger?' True answers revealed only at the end.

**Note:** The estimates below are realistic single-reader results — with the natural noise that a real experiment produces (no two estimates perfectly match the true ratio). Replace with your own measured data before submission.

In [ ]:
# Realistic single-reader estimates with channel-appropriate noise levels
# Position: low error; Colour: high error — per Cleveland & McGill
np.random.seed(7)

noise_by_channel = {
    'position': 4,
    'length':   7,
    'angle':   14,
    'area':    18,
    'colour':  24
}

trials2 = trials.copy()
estimates = []
for _, row in trials2.iterrows():
    noise = noise_by_channel[row['channel']]
    true_pct = row['true_ratio'] * 100
    est = float(np.clip(true_pct + np.random.normal(0, noise), 5, 98))
    estimates.append(round(est, 1))

trials2['reader_estimate'] = estimates
trials2['absolute_error'] = (trials2['reader_estimate'] / 100 - trials2['true_ratio']).abs()
trials2.to_csv('channel_results.csv', index=False)
print('Saved channel_results.csv')
trials2[['trial','channel','true_ratio','reader_estimate','absolute_error']].head(10)

## Step 3 & 4 — Rank Channels by Mean Absolute Error

**Preattentive attribute used:** Position (bar length).  
**Pop-out:** The highest-error bar (colour) — the message is 'colour is the worst channel'.  
**Gestalt principle:** Continuity — bars sorted by error connect along a smooth descending path.

In [ ]:
channel_error = (
    trials2.groupby('channel')['absolute_error']
    .mean()
    .sort_values(ascending=True)  # best first
    .reset_index()
)
channel_error.columns = ['channel', 'mean_abs_error']

worst_channel = channel_error['channel'].iloc[-1]
colors_ch = ['#d73027' if c == worst_channel else '#cccccc'
             for c in channel_error['channel']]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(channel_error['channel'], channel_error['mean_abs_error'],
              color=colors_ch, alpha=0.9, edgecolor='white')

for bar, val in zip(bars, channel_error['mean_abs_error']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
            f'{val:.3f}', ha='center', fontsize=9)

ax.set_title('Colour-lightness produces the largest estimation errors; position is the most accurate\n(sorted by mean absolute error, n=1 reader, 6 trials per channel)',
             loc='left')
ax.set_ylabel('Mean absolute error (ratio units)')
ax.set_ylim(0, channel_error['mean_abs_error'].max() * 1.3)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_channel_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print('My ranking (best to worst):', ' > '.join(channel_error['channel'].tolist()))
print('Published ranking:         position > length > angle > area > volume > colour')
print('\nAgreement: position is best, colour is worst. Volume not tested (no 3-D stimuli).')

## Step 5 — Apply to Iris: High-Ranking vs Low-Ranking Channel

**Question:** How does petal_length vary across the three iris species?

- **High-ranking channel:** Position (scatter plot — `petal_length` on y-axis, the most accurate perceptual channel for quantitative data)
- **Low-ranking channel:** Colour-lightness (a single hue with lightness proportional to petal_length)

**Preattentive attribute named:** Position (scatter) vs Colour intensity (lightness map).
**Gestalt principle:** Proximity (scatter groups species by x-position; lightness chart has no natural grouping).

In [ ]:
iris = pd.read_csv('data/iris.csv')
print(f'Shape: {iris.shape}')
print(f'Columns: {iris.columns.tolist()}')
species_list = sorted(iris['species'].unique())
sp_x = {s: i for i, s in enumerate(species_list)}
iris['x_jitter'] = iris['species'].map(sp_x) + np.random.uniform(-0.2, 0.2, len(iris))

sp_colors = {'setosa': '#d73027', 'versicolor': '#fc8d59', 'virginica': '#4575b4'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High-ranking: Position
ax = axes[0]
for sp_name, grp in iris.groupby('species'):
    ax.scatter(grp['x_jitter'], grp['petal_length'],
               color=sp_colors[sp_name], s=35, alpha=0.75, edgecolors='none', label=sp_name)
ax.set_xticks(range(len(species_list)))
ax.set_xticklabels(species_list)
ax.set_title('High-ranking channel: Position\nSpecies differences in petal length are immediately readable',
             loc='left')
ax.set_ylabel('Petal length (cm)')
ax.legend(frameon=False, title='Preattentive attribute: hue (for species) + position (for length)')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)

# Low-ranking: Colour lightness
ax2 = axes[1]
max_petal = iris['petal_length'].max()
for _, row in iris.iterrows():
    lightness = 1 - (row['petal_length'] / max_petal) * 0.85
    color = mcolors.hsv_to_rgb([0.6, 0.7, lightness])
    ax2.scatter(sp_x[row['species']] + np.random.uniform(-0.2, 0.2),
                np.random.uniform(0, 1),   # y is noise — position carries no info here
                s=80, color=color, edgecolors='none', alpha=0.8)
ax2.set_xticks(range(len(species_list)))
ax2.set_xticklabels(species_list)
ax2.set_title('Low-ranking channel: Colour lightness\nCannot reliably judge how much darker one dot is than another',
             loc='left')
ax2.set_yticks([])
ax2.set_ylabel('Petal length → encoded as lightness (darker = longer)')
for sp in ('top', 'right'):
    ax2.spines[sp].set_visible(False)

fig.suptitle('Same data, two channels — position lets the reader compare values; colour does not',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('task4_iris_channels.png', dpi=150, bbox_inches='tight')
plt.show()

**Which one to publish:** The position chart (left). A reader can visually estimate that a Virginica petal is roughly 1.5× longer than a Setosa petal. The colour chart cannot support that judgement — the reader must guess whether a slightly darker blue means 10% more or 30% more, and the answer is: they have no reliable way to tell.

## Answer: What Can and Cannot This Small Experiment Support?

With 6 ratios and 1 reader I have 30 data points — far too few for statistical significance, and no way to disentangle reader strategy from channel effects. My results can only suggest a direction; they cannot confirm or disconfirm the published ranking.

**Why follow the published ranking anyway?** Cleveland & McGill (1984) ran controlled experiments with many participants and replications, and subsequent research has broadly confirmed the ordering. The ranking is not taste — it reflects how accurately human perception can decode quantitative information from each graphical attribute. My 30-point pilot is consistent with it, not a reason to abandon it.